# CrewAI + Local Ollama

**Flow**

`User → Main Orchestrator → Sub-Agents → Main Orchestrator → User`

- **Main Agent:** orchestrates the work and delegates tasks to the specialist agents; it has no tools.
- **Weather Sub-Agent:** uses the Open-Meteo tool.
- **Knowledge Sub-Agent:** uses the Wikipedia tool.
- **Local LLM:** Ollama inside Google Colab.

In [2]:
# Install zstd first — required by the current Ollama installer
!apt-get update -qq
!apt-get install -y -qq zstd

# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# Verify installation
!which ollama
!ollama --version

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package zstd.
(Reading database ... 122809 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
/usr/local/bin/ollama


In [3]:
!pip install -q requests==2.32.4
# CrewAI uses LiteLLM for many model-provider integrations.
!pip -q install -U crewai litellm requests

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.4/202.4 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.4/27.4 MB 68.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.

In [4]:
import subprocess, time

subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.STDOUT
)

# ----------------------
# poll instead of a fixed sleep, more reliable than the original 3s sleep
import requests
for _ in range(30):
    try:
        r = requests.get("http://localhost:11434/api/tags", timeout=2)
        if r.status_code == 200:
            print("Ollama is up")
            break
    except requests.exceptions.ConnectionError:
        pass
    time.sleep(1)
else:
    print("Ollama never came up — check for port conflicts or install issues")

MODEL_NAME = "qwen2.5:3b"

!ollama pull {MODEL_NAME}

Ollama is up



In [15]:
import requests

response = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": MODEL_NAME,
        "prompt": "Reply with exactly: Local LLM is working.",
        "stream": False,
    },
    timeout=120,
)

print(response.json()["response"])

Local LLM is working.


## External tools

- Open-Meteo → weather
- Wikipedia → factual background

**Architecture:**

`User → Main Orchestrator → Weather Specialist / Knowledge Specialist → Main Orchestrator → User`


In [5]:
import requests
from crewai.tools import tool

@tool("Weather Tool")
def weather_tool(city: str) -> str:
    """Get current weather for a city using Open-Meteo."""
    try:
        geo = requests.get(
            "https://geocoding-api.open-meteo.com/v1/search",
            params={"name": city, "count": 1, "language": "en", "format": "json"},
            timeout=15,
        ).json()

        if not geo.get("results"):
            return f"Could not find the city: {city}"

        place = geo["results"][0]
        forecast = requests.get(
            "https://api.open-meteo.com/v1/forecast",
            params={
                "latitude": place["latitude"],
                "longitude": place["longitude"],
                "current": "temperature_2m,relative_humidity_2m,weather_code,wind_speed_10m",
                "timezone": "auto",
            },
            timeout=15,
        ).json()

        c = forecast["current"]
        return (
            f"Location: {place.get('name', city)}, {place.get('country', '')}\n"
            f"Temperature: {c['temperature_2m']} °C\n"
            f"Humidity: {c['relative_humidity_2m']}%\n"
            f"Wind: {c['wind_speed_10m']} km/h\n"
            f"Weather code: {c['weather_code']}"
        )
    except Exception as e:
        return f"Weather tool error: {e}"

@tool("Wikipedia Tool")
def wikipedia_tool(topic: str) -> str:
    """Get a concise Wikipedia summary for a topic."""
    try:
        url = (
            "https://en.wikipedia.org/api/rest_v1/page/summary/"
            + requests.utils.quote(topic.replace(" ", "_"))
        )
        r = requests.get(url, timeout=15)
        if r.status_code != 200:
            return f"No Wikipedia page found for: {topic}"
        return r.json().get("extract", f"No summary found for {topic}")
    except Exception as e:
        return f"Wikipedia tool error: {e}"

In [6]:
from crewai import LLM, Agent, Task, Crew, Process

local_llm = LLM(
    model=f"ollama/{MODEL_NAME}",
    base_url="http://localhost:11434",
    temperature=0,
)

print("CrewAI local LLM configured:", MODEL_NAME)

CrewAI local LLM configured: qwen2.5:3b


## Two specialist agents + one orchestrating manager


In [7]:
weather_agent = Agent(
    role="Weather Specialist",
    goal="Get accurate current weather for the requested location.",
    backstory=(
        "You are a weather specialist. You always use the weather tool "
        "instead of guessing current weather."
    ),
    tools=[weather_tool],
    llm=local_llm,
    verbose=True,
    allow_delegation=False,
)

knowledge_agent = Agent(
    role="Knowledge Specialist",
    goal="Provide concise factual background about the requested place or topic.",
    backstory=(
        "You are a factual research specialist. You use Wikipedia when "
        "you need factual background and do not invent missing information."
    ),
    tools=[wikipedia_tool],
    llm=local_llm,
    verbose=True,
    allow_delegation=False,
)

In [8]:
main_agent = Agent(
    role="Main Orchestrator",
    goal=(
        "Understand the user's request, decide which specialist should handle "
        "each part of the work, delegate those tasks, and synthesize the results "
        "into one useful answer."
    ),
    backstory=(
        "You are the lead agent coordinating a small team. "
        "For travel/location questions, delegate current weather to the Weather "
        "Specialist and factual background to the Knowledge Specialist. "
        "You do not call external tools yourself; you orchestrate the specialists."
    ),
    llm=local_llm,
    verbose=True,
    allow_delegation=True,
)


In [9]:
# !curl -s http://localhost:11434/api/tags
!curl -v http://localhost:11434/api/tags

* Host localhost:11434 was resolved.
* IPv6: ::1
* IPv4: 127.0.0.1
*   Trying [::1]:11434...
* connect to ::1 port 11434 from ::1 port 48688 failed: Connection refused
*   Trying 127.0.0.1:11434...
* Connected to localhost (127.0.0.1) port 11434
> GET /api/tags HTTP/1.1
> Host: localhost:11434
> User-Agent: curl/8.5.0
> Accept: */*
> 
< HTTP/1.1 200 OK
< Content-Type: application/json; charset=utf-8
< Date: Wed, 23 Sep 2026 13:01:57 GMT
< Content-Length: 418
< 
* Connection #0 to host localhost left intact
{"models":[{"name":"qwen2.5:3b","model":"qwen2.5:3b","modified_at":"2026-09-23T13:01:46.01514576Z","size":1929912432,"digest":"357c53fb659c5076de1d65ccb0b397446227b71a42be9d1603d46168015c9e4b","details":{"parent_model":"","format":"gguf","family":"qwen2","families":["qwen2"],"parameter_size":"3.1B","quantization_level":"Q4_K_M","context_length":32768,"embedding_length":2048},"capabilities":["completion","tools"]}]}

In [20]:
user_request = """
I am planning to visit Mysuru. Tell me the current weather and give me a
short introduction to Mysuru that would help a first-time visitor.
"""

main_task = Task(
    description=(
        "Handle the user's request as the Main Orchestrator. Break the request "
        "into appropriate pieces and delegate the weather work to the Weather "
        "Specialist and factual background work to the Knowledge Specialist. "
        "Review their outputs and synthesize one final answer. "
        f"USER REQUEST:\n{user_request}"
    ),
    expected_output=(
        "A concise final answer that combines the specialist results, clearly "
        "separating the weather and factual background when useful."
    ),
    # agent=main_agent,
)

demo_crew = Crew(
    agents=[weather_agent, knowledge_agent],
    tasks=[main_task],
    process=Process.hierarchical,
    manager_agent=main_agent,
    verbose=True
)

result = await demo_crew.kickoff_async()
# result = demo_crew.kickoff()

print("\n" + "=" * 70)
print("FINAL ANSWER")
print("=" * 70)
print(result)


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 1128b48b-083c-41f2-8a02-e5e90d46aca9                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Handle the user's request as the Main Orchestrator. Break the request into appropriate pieces and        │
│  delegate the weather work to the Weather Specialist and factual background work to the Knowledge Specialist.   │
│  Review their outputs and synthesize one final answer. USER REQUEST:                                            │
│                                                                                                                 │
│  I am planning to visit Mysuru. Tell me the current weather and give me a                                       │
│  short introduction to Mysuru that would help a first-time visitor.                                             │
│                                                                                                                 │
│  ID: 01e97088-9825-49ea-b168-539d4e004be5                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Main Orchestrator                                                                                       │
│                                                                                                                 │
│  Task: Handle the user's request as the Main Orchestrator. Break the request into appropriate pieces and        │
│  delegate the weather work to the Weather Specialist and factual background work to the Knowledge Specialist.   │
│  Review their outputs and synthesize one final answer. USER REQUEST:                                            │
│                                                                                                                 │
│  I am planning to visit Mysuru. Tell me the current weather and give me a                                       │
│  short introduction to Mysuru that would help a first-time visitor.                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: ask_question_to_coworker                                                                                 │
│  Args: {'context': "Current Task: Handle the user's request as the Main Orchestrator. Break the request into    │
│  appropriate pieces and delegate the weather work to the Weather Specialist and factual background ...          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#2) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: ask_question_to_coworker                                                                                 │
│  Args: {'context': "Current Task: Handle the user's request as the Main Orchestrator. Break the request into    │
│  appropriate pieces and delegate the weather work to the Weather Specialist and factual background ...          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Main Orchestrator                                                                                       │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  The current weather in Mysuru, India is as follows: the temperature is 28.6 °C, the humidity is at 51%, the    │
│  wind speed is 22.1 km/h, and the weather condition is described by code 3.                                     │
│                                                                                                                 │
│  For a short introduction to Mysuru, which is likely the name of Mysore in Karnataka, India, I would suggest    │
│  the following:                                                                                                 │
│                                                                                                                 │
│  Mysore, also known as Mysuru, is a city in the southern Indian state of Karnataka. It is known for its rich    │
│  history, cultural heritage, and beautiful palaces. The city is home to several important temples, including    │
│  the Chamundi Hills, and is a popular tourist destination. Mysore is also known for its traditional silk        │
│  sarees and its annual Mysore Dasara festival, which features elaborate processions and temple fairs. The city  │
│  has a significant population of the Mysore Kannada speaking community, and its cuisine is influenced by both   │
│  Kannada and Kannada-speaking influences.                                                                       │
│                                                                                                                 │
│  Would you like any further information?                                                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Handle the user's request as the Main Orchestrator. Break the request into appropriate pieces and        │
│  delegate the weather work to the Weather Specialist and factual background work to the Knowledge Specialist.   │
│  Review their outputs and synthesize one final answer. USER REQUEST:                                            │
│                                                                                                                 │
│  I am planning to visit Mysuru. Tell me the current weather and give me a                                       │
│  short introduction to Mysuru that would help a first-time visitor.                                             │
│                                                                                                                 │
│  Agent: Main Orchestrator                                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 1128b48b-083c-41f2-8a02-e5e90d46aca9                                                                       │
│  Final Output: The current weather in Mysuru, India is as follows: the temperature is 28.6 °C, the humidity is  │
│  at 51%, the wind speed is 22.1 km/h, and the weather condition is described by code 3.                         │
│                                                                                                                 │
│  For a short introduction to Mysuru, which is likely the name of Mysore in Karnataka, India, I would suggest    │
│  the following:                                                                                                 │
│                                                                                                                 │
│  Mysore, also known as Mysuru, is a city in the southern Indian state of Karnataka. It is known for its rich    │
│  history, cultural heritage, and beautiful palaces. The city is home to several important temples, including    │
│  the Chamundi Hills, and is a popular tourist destination. Mysore is also known for its traditional silk        │
│  sarees and its annual Mysore Dasara festival, which features elaborate processions and temple fairs. The city  │
│  has a significant population of the Mysore Kannada speaking community, and its cuisine is influenced by both   │
│  Kannada and Kannada-speaking influences.                                                                       │
│                                                                                                                 │
│  Would you like any further information?                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


FINAL ANSWER
The current weather in Mysuru, India is as follows: the temperature is 28.6 °C, the humidity is at 51%, the wind speed is 22.1 km/h, and the weather condition is described by code 3.

For a short introduction to Mysuru, which is likely the name of Mysore in Karnataka, India, I would suggest the following:

Mysore, also known as Mysuru, is a city in the southern Indian state of Karnataka. It is known for its rich history, cultural heritage, and beautiful palaces. The city is home to several important temples, including the Chamundi Hills, and is a popular tourist destination. Mysore is also known for its traditional silk sarees and its annual Mysore Dasara festival, which features elaborate processions and temple fairs. The city has a significant population of the Mysore Kannada speaking community, and its cuisine is influenced by both Kannada and Kannada-speaking influences.

Would you like any further information?


In [ ]:
user_request = input("Enter a request for the Main Agent: ")

main_task = Task(
    description=(
        "Act as the Main Orchestrator. Analyze the request, delegate each "
        "relevant part to the appropriate specialist agent, review their results, "
        "and synthesize the final answer. Do not call external tools yourself."
        "USER REQUEST:\n" + user_request
    ),
    expected_output="A concise, factual synthesized answer based on the specialist agents' work.",
)

demo_crew = Crew(
    agents=[weather_agent, knowledge_agent],
    tasks=[main_task],
    process=Process.hierarchical,
    manager_agent=main_agent,
    verbose=True,
)

result = await demo_crew.kickoff_async()
print("\n" + "=" * 70)
print("FINAL ANSWER")
print("=" * 70)
print(result)
